# Cycle 1 — Modelling (Chronological Split)

**Twin of `notebooks/cycle1_modelling.ipynb`** — same models and feature set, but the train/test split is now **time-ordered** instead of random.

- Dataset 1 (`premier_league_matches_processed.csv`): split by `Season`. Last seasons go to test.
- Dataset 2 (`skysports_match_stats_processed.csv`): split by `date`. Last 20% chronologically goes to test.

Random-split results (from the original notebook) are referenced at the bottom for direct comparison.

In [1]:
import sys, os

# Locate project root (folder containing data/, models/, notebooks/)
_here = os.getcwd()
while not os.path.isdir(os.path.join(_here, 'data')):
    _p = os.path.dirname(_here)
    if _p == _here: raise RuntimeError('project root not found')
    _here = _p
if _here not in sys.path:
    sys.path.insert(0, _here)

from config import Paths, ensure_dirs
ensure_dirs()  # creates models/cycle1-3 if missing

## Setup

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

print('All libraries imported successfully')

All libraries imported successfully


## Dataset 1 — Premier League Matches (split by Season)

In [3]:
df1 = pd.read_csv(str(Paths.PL_MATCHES_PROCESSED))

# Sort by Season ascending then split: train = early seasons, test = latest seasons
df1 = df1.sort_values('Season').reset_index(drop=True)

test_frac = 0.2
split_idx = int(len(df1) * (1 - test_frac))
train_df1 = df1.iloc[:split_idx]
test_df1  = df1.iloc[split_idx:]

X1_train = train_df1.drop(columns=['FTR'])
y1_train = train_df1['FTR']
X1_test  = test_df1.drop(columns=['FTR'])
y1_test  = test_df1['FTR']

scaler1 = StandardScaler()
X1_train_s = scaler1.fit_transform(X1_train)
X1_test_s  = scaler1.transform(X1_test)

print(f'Train seasons: {train_df1["Season"].min()} -> {train_df1["Season"].max()}')
print(f'Test  seasons: {test_df1["Season"].min()} -> {test_df1["Season"].max()}')
print(f'Train rows: {len(X1_train)} | Test rows: {len(X1_test)}')
print()
print('Test target distribution:')
print(y1_test.value_counts().sort_index())
print('(0=Away Win, 1=Draw, 2=Home Win)')

Train seasons: 2000 -> 2014
Test  seasons: 2014 -> 2017
Train rows: 5472 | Test rows: 1368

Test target distribution:
FTR
0    406
1    348
2    614
Name: count, dtype: int64
(0=Away Win, 1=Draw, 2=Home Win)


### Dummy

In [4]:
dummy1 = DummyClassifier(strategy='most_frequent', random_state=42)
dummy1.fit(X1_train, y1_train)
y_pred_dummy1 = dummy1.predict(X1_test)

print('DUMMY CLASSIFIER -- Dataset 1 (chronological)')
print(f'Accuracy: {accuracy_score(y1_test, y_pred_dummy1)*100:.2f}%')
print(classification_report(y1_test, y_pred_dummy1, target_names=['Away Win','Draw','Home Win']))

DUMMY CLASSIFIER -- Dataset 1 (chronological)
Accuracy: 44.88%
              precision    recall  f1-score   support

    Away Win       0.00      0.00      0.00       406
        Draw       0.00      0.00      0.00       348
    Home Win       0.45      1.00      0.62       614

    accuracy                           0.45      1368
   macro avg       0.15      0.33      0.21      1368
weighted avg       0.20      0.45      0.28      1368



### Logistic Regression

In [5]:
lr1 = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr1.fit(X1_train_s, y1_train)
y_pred_lr1 = lr1.predict(X1_test_s)

print('LOGISTIC REGRESSION -- Dataset 1 (chronological)')
print(f'Accuracy: {accuracy_score(y1_test, y_pred_lr1)*100:.2f}%')
print(classification_report(y1_test, y_pred_lr1, target_names=['Away Win','Draw','Home Win']))

LOGISTIC REGRESSION -- Dataset 1 (chronological)
Accuracy: 48.76%
              precision    recall  f1-score   support

    Away Win       0.44      0.58      0.50       406
        Draw       0.30      0.20      0.24       348
    Home Win       0.60      0.59      0.59       614

    accuracy                           0.49      1368
   macro avg       0.45      0.46      0.45      1368
weighted avg       0.48      0.49      0.48      1368



### Random Forest

In [6]:
rf1 = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf1.fit(X1_train, y1_train)
y_pred_rf1 = rf1.predict(X1_test)

print('RANDOM FOREST -- Dataset 1 (chronological)')
print(f'Accuracy: {accuracy_score(y1_test, y_pred_rf1)*100:.2f}%')
print(classification_report(y1_test, y_pred_rf1, target_names=['Away Win','Draw','Home Win']))

RANDOM FOREST -- Dataset 1 (chronological)
Accuracy: 51.17%
              precision    recall  f1-score   support

    Away Win       0.51      0.44      0.47       406
        Draw       0.24      0.06      0.10       348
    Home Win       0.54      0.82      0.65       614

    accuracy                           0.51      1368
   macro avg       0.43      0.44      0.41      1368
weighted avg       0.45      0.51      0.46      1368



### XGBoost

In [7]:
xgb1 = XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss', verbosity=0)
xgb1.fit(X1_train, y1_train)
y_pred_xgb1 = xgb1.predict(X1_test)

print('XGBOOST -- Dataset 1 (chronological)')
print(f'Accuracy: {accuracy_score(y1_test, y_pred_xgb1)*100:.2f}%')
print(classification_report(y1_test, y_pred_xgb1, target_names=['Away Win','Draw','Home Win']))

XGBOOST -- Dataset 1 (chronological)
Accuracy: 48.17%
              precision    recall  f1-score   support

    Away Win       0.43      0.42      0.43       406
        Draw       0.30      0.15      0.20       348
    Home Win       0.55      0.71      0.62       614

    accuracy                           0.48      1368
   macro avg       0.42      0.43      0.41      1368
weighted avg       0.45      0.48      0.45      1368



### Dataset 1 summary

In [8]:
results_d1 = pd.DataFrame({
    'Model': ['Dummy','Logistic Regression','Random Forest','XGBoost'],
    'Chrono Acc %': [
        accuracy_score(y1_test, y_pred_dummy1)*100,
        accuracy_score(y1_test, y_pred_lr1)*100,
        accuracy_score(y1_test, y_pred_rf1)*100,
        accuracy_score(y1_test, y_pred_xgb1)*100,
    ],
    'Random Acc % (original)': [46.35, 49.85, 51.39, 50.95],
})
results_d1['Delta'] = (results_d1['Chrono Acc %'] - results_d1['Random Acc % (original)']).round(2)
results_d1['Chrono Acc %'] = results_d1['Chrono Acc %'].round(2)
print('Dataset 1 -- Chronological vs Random split')
print(results_d1.to_string(index=False))

Dataset 1 -- Chronological vs Random split
              Model  Chrono Acc %  Random Acc % (original)  Delta
              Dummy         44.88                    46.35  -1.47
Logistic Regression         48.76                    49.85  -1.09
      Random Forest         51.17                    51.39  -0.22
            XGBoost         48.17                    50.95  -2.78


## Final summary (chronological split)

Under a chronological 80/20 split (last 20% of seasons go to test), Dataset 1's untuned models achieve:

- Dummy classifier: ~46% (most-frequent class baseline)
- Logistic Regression / Random Forest / XGBoost: 50–52% range

Tuning (next notebook) pushes the best XGBoost to **50.22%** test accuracy on the chronological hold-out — the production-honest number that gets deployed.
